In [77]:
import cupy as cp
import numpy as np
import math
import random
import torch
import gc

from numba import cuda, types, njit
from numba.cuda import jit as cjit, random as crandom
from numpy import ndarray as CPUArray
from typing import Union

GPUArray = Union[cuda.devicearray.DeviceNDArray, cp.ndarray]

In [78]:
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DTYPE   = torch.float32

In [79]:
def calc(tasks: tuple[int], threads: int):
    ndim = len(tasks)
    bpg: tuple[int] = (math.ceil((tasks[0]) / threads),)
    tpb = (threads,)
    if ndim == 1:
        return bpg + tpb
    else:
        for dim_idx in range(1, ndim):
            t = threads
            if dim_idx == 1:
                t = min(10, t)
            if dim_idx == 2:
                t = min(10, t)
                # print(tpb, tasks[dim_idx], t, tasks[dim_idx] / t)
            bpg += (math.ceil((tasks[dim_idx]) / t * 1),)
            if len(tasks) > 1:
                tpb += (t,)
        return bpg, tpb

def calc_grid(*block_sizes: int, tpb=4):
    for _ in range(3-len(block_sizes)):
        block_sizes += (1,)
    return calc(block_sizes, tpb)

In [80]:
torch.pow(torch.tensor([2.0 ** 64]), 1/4)

tensor([65536.])

In [81]:
SEED = random.randint(0, 1e6)
def get_rng_states(kernel_shape: tuple, seed: int = None, xoro=False, normal=False, use_cuda=False):
    if seed is None:
        seed = SEED
    threads_total = int(np.prod([np.prod(v) for v in kernel_shape]))
    if xoro:
        rng_states = crandom.create_xoroshiro128p_states(threads_total, seed=seed)
    else:
        if normal:
            rng_states = np.random.normal(size=threads_total) if not use_cuda else cp.random.normal(size=threads_total)
        else:
            rng_states = np.random.rand(threads_total) if not use_cuda else cp.random.rand(threads_total)
        # rng_states = rng_states.view(np.float32)
    return rng_states, threads_total

In [82]:
def delete_array(array: GPUArray):
    array = array.copy_to_host()
    del array

In [83]:
# from ModifiedNEAT.cuda.functional.generation import

In [84]:
@njit
def clamp(value: float, minimum: float, maximum: float):
    return min(max(value, minimum), maximum)

In [108]:
@cjit(device=True)
def prob(states: GPUArray, index: int):
    return states[index]

@cjit(device=True)
def prob_xoro(states: GPUArray, index: int):
    return crandom.xoroshiro128p_uniform_float64(states, index)

In [109]:
@cjit
def check_probs(probs: GPUArray, randomizer: GPUArray):
    x, y, z = cuda.grid(3)
    limits = probs.shape
    s_x, s_y, s_z = cuda.gridsize(3)
    if x < limits[0] and y < limits[1] and z < limits[2]:
        # Linearized thread index
        rng_index = (z * s_y * s_x) + (y * s_x) + x
        probs[x, y, z] = prob(randomizer, rng_index)

@cjit
def check_probs_xoro(probs: GPUArray, randomizer: GPUArray):
    x, y, z = cuda.grid(3)
    limits = probs.shape
    s_x, s_y, s_z = cuda.gridsize(3)
    if x < limits[0] and y < limits[1] and z < limits[2]:
        # Linearized thread index
        rng_index = (z * s_y * s_x) + (y * s_x) + x
        probs[x, y, z] = prob_xoro(randomizer, rng_index)

In [110]:
test_shape = (100, 128, 128)
test_tpb = 1

In [111]:
gc.collect()

266

In [114]:
%%timeit -n 10 -r 3
test_prob_tensor = torch.zeros(*test_shape, device=DEVICE, dtype=DTYPE)
test_prob_array = test_prob_tensor.cpu().numpy()
test_rng_states = get_rng_states(test_shape, xoro=True, seed=1)[0]
check_probs_xoro[*calc_grid(*test_shape, tpb=test_tpb)](test_prob_array, test_rng_states)
test_rng_states = test_rng_states.copy_to_host()
test_prob_tensor.copy_(torch.tensor(test_prob_array, device=test_prob_tensor.device))
del test_rng_states, test_prob_array
# test_prob_tensor[0], (test_prob_tensor <= 0.1).count_nonzero(), (test_prob_tensor >= 0.9).count_nonzero()

348 ms ± 37.1 ms per loop (mean ± std. dev. of 3 runs, 10 loops each)


In [115]:
gc.collect()

8

In [118]:
%%timeit -n 10 -r 10
test_prob_tensor = torch.zeros(*test_shape, device=DEVICE, dtype=DTYPE)
test_prob_array = cuda.to_device(test_prob_tensor.cpu().numpy())
test_rng_states = cuda.to_device(get_rng_states(test_shape, xoro=False, seed=1)[0])
check_probs[*calc_grid(*test_shape, tpb=test_tpb)](test_prob_array, test_rng_states)
test_prob_array = test_prob_array.copy_to_host()
test_rng_states = test_rng_states.copy_to_host()
test_prob_tensor.copy_(torch.tensor(test_prob_array, device=test_prob_tensor.device))
del test_rng_states, test_prob_array
# test_prob_tensor[0], (test_prob_tensor <= 0.1).count_nonzero(), (test_prob_tensor >= 0.9).count_nonzero()

29.6 ms ± 15.4 ms per loop (mean ± std. dev. of 10 runs, 10 loops each)


In [119]:
gc.collect()

0

In [122]:
%%timeit -n 10 -r 10
test_prob_tensor = torch.zeros(*test_shape, device=DEVICE, dtype=DTYPE)
test_prob_array = test_prob_tensor.cpu().numpy()
test_rng_states = get_rng_states(test_shape, xoro=False, seed=1)[0]
check_probs[*calc_grid(*test_shape, tpb=test_tpb)](test_prob_array, test_rng_states)
test_prob_tensor.copy_(torch.tensor(test_prob_array, device=test_prob_tensor.device))
del test_rng_states, test_prob_array
# test_prob_tensor[0], (test_prob_tensor <= 0.1).count_nonzero(), (test_prob_tensor >= 0.9).count_nonzero()

26.9 ms ± 15.2 ms per loop (mean ± std. dev. of 10 runs, 10 loops each)


In [121]:
gc.collect()

107

In [431]:
test_shape = (100, 512, 512)

In [441]:
%%timeit -n 10 -r 10
test_prob_tensor = torch.zeros(*test_shape, device=DEVICE, dtype=DTYPE)
test_prob_array = cp.asarray(test_prob_tensor)
test_rng_states = get_rng_states(test_shape, use_cuda=True, seed=1)[0]
check_probs[*calc_grid(*test_shape, tpb=test_tpb)](test_prob_array, test_rng_states)
test_prob_tensor.copy_(torch.from_dlpack(test_prob_array))
del test_rng_states, test_prob_array
cp.get_default_memory_pool().free_all_blocks()
# test_prob_tensor[0], (test_prob_tensor <= 0.1).count_nonzero(), (test_prob_tensor >= 0.9).count_nonzero()

47.2 ms ± 5.86 ms per loop (mean ± std. dev. of 10 runs, 10 loops each)


In [137]:
gc.collect()

99

In [138]:
import time as clock

In [139]:
test_create = cp.random.randn(6000, 4, 64, 64)
clock.sleep(3)
del test_create
cp.get_default_memory_pool().free_all_blocks()

In [140]:
gc.collect()

1675

In [331]:
@cjit(device=True)
def mix_values(value0: float, value1: float, states: GPUArray, index: int):
    if prob(states, index) < 0.50:
        return value0
    else:
        return value1

@cjit
def crossover(genome_a: GPUArray, genome_b: GPUArray, genome_c: GPUArray, randomizer: GPUArray, loops: int):
    x, y, z = cuda.grid(3)
    limits = genome_c.shape
    s_x, s_y, s_z = cuda.gridsize(3)
    if x < limits[0] and y < limits[1] and z < limits[2]:
        # Linearized thread index
        rng_index = (z * s_y * s_x) + (y * s_x) + x
        for _ in range(loops):
            genome_c[x, y, z] = mix_values(genome_a[x, y, z], genome_b[x, y, z], randomizer, rng_index)
            cuda.syncthreads()

In [332]:
INPUTS = 512
OUTPUTS = 512
KERNEL = 3
PARAMETER = (INPUTS, OUTPUTS, KERNEL)

In [343]:
# %%timeit -n 10 -r 3
source: tuple[GPUArray, ...] = [cp.asarray(g.squeeze(0)) for g in torch.chunk(torch.randn(3, *PARAMETER, device=DEVICE, dtype=DTYPE), 3)]
parentA, parentB, child = source
rng_states = get_rng_states(PARAMETER, use_cuda=True, seed=1)[0]
crossover[*calc_grid(*PARAMETER, tpb=1)](parentA, parentB, child, rng_states, 1)
parentA, parentB, child = parentA.get(), parentB.get(), child.get()
cp.get_default_memory_pool().free_all_blocks()
child[:2, :4, :4], np.count_nonzero(child==parentA), np.count_nonzero(child==parentB)

(array([[[ 0.49131224, -2.9303944 ,  0.2602116 ],
         [ 0.44273973,  0.7227036 ,  0.31800285],
         [ 0.15365006, -0.56186163, -0.05906649],
         [ 0.86091727, -1.1154133 ,  0.7775614 ]],
 
        [[-0.32524347,  0.27898982, -0.6320097 ],
         [ 0.6893887 ,  1.8960952 ,  1.6814655 ],
         [-0.4143476 , -0.21151635, -0.64629835],
         [ 2.1176815 ,  1.3280687 , -0.24677461]]], dtype=float32),
 394155,
 392277)

In [389]:
%%timeit -n 10 -r 3
test_array = cp.random.uniform(size=10)
test_array.get()


168 μs ± 77.2 μs per loop (mean ± std. dev. of 3 runs, 10 loops each)


In [400]:
%%timeit -n 10 -r 3
test_array = cp.random.rand(10)
test_array.get()

134 μs ± 48.8 μs per loop (mean ± std. dev. of 3 runs, 10 loops each)


In [428]:
%%timeit -n 100 -r 3
test_tensor = torch.randn(512, 512, 64, device=DEVICE)
test_tensor = test_tensor.reshape(-1)
del test_tensor
torch.cuda.empty_cache()

4.85 ms ± 2.66 ms per loop (mean ± std. dev. of 3 runs, 100 loops each)


In [430]:
%%timeit -n 100 -r 3
test_tensor = cp.random.randn(512, 512, 64)
test_tensor = test_tensor.reshape(-1)
del test_tensor
cp.get_default_memory_pool().free_all_blocks()

15.3 ms ± 1.25 ms per loop (mean ± std. dev. of 3 runs, 100 loops each)
